# import libraries

In [61]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import sqlite3

# import dataset

In [62]:
df = pd.read_excel('customer_churn_data_new.xlsx', sheet_name=None)

In [63]:
df

{'db_customer':       customerid    name country        state  gender        dob    interests  \
 0     0002-ORFBO  keshav   India  Maharashtra    Male 1982-04-12       travel   
 1     0003-MKNFE  raghav   India    Karnataka    Male 1995-11-23          NaN   
 2     0004-TLHLJ  lalita   India        Delhi  Female 1978-02-15        movie   
 3     0011-IGKFF   mohan   India     Nagaland    Male 2001-08-30          NaN   
 4     0013-EXCHZ    mira   India        Delhi  Female 1990-05-05        drama   
 ...          ...     ...     ...          ...     ...        ...          ...   
 6016  1159-SQVTL  Sanjay   India  West Bengal  Female 1960-04-19        music   
 6017  9384-ZXAED   Ayaan   India       Punjab  Female 1970-01-18      fashion   
 6018  4003-SFEGL    Riya   India       Kerala  Female 1998-01-14        music   
 6019  9591-NAXJL   Priya   India   Tamil Nadu  Female 1991-05-05        music   
 6020  6448-TVCRQ    Riya   India        Bihar    Male 1987-10-22  photography   
 

In [64]:
print(df.keys())

dict_keys(['db_customer', 'db_subscription', 'db_support'])


In [65]:
df_customer = df['db_customer']
df_subscription = df['db_subscription']
df_support = df['db_support']

In [66]:
df_customer.head()

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12,travel,NaN
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23,NaN,NaN
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15,movie,NaN
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30,NaN,NaN
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05,drama,NaN


In [67]:
df_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6021 entries, 0 to 6020
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   customerid  6021 non-null   object        
 1   name        6021 non-null   object        
 2   country     6018 non-null   object        
 3   state       6021 non-null   object        
 4   gender      6021 non-null   object        
 5   dob         6021 non-null   datetime64[ns]
 6   interests   5482 non-null   object        
 7   pincode     6000 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 376.4+ KB


In [68]:
df_customer.describe()

,dob,pincode
count,6021,6000.000000
mean,1982-07-02 17:32:19.013452928,504560.283833
min,1960-01-01 00:00:00,110039.000000
25%,1971-06-19 00:00:00,307043.250000
50%,1982-05-27 00:00:00,507631.500000
75%,1993-09-15 00:00:00,701552.000000
max,2004-12-26 00:00:00,899928.000000
std,NaN,228871.833628


# data cleanning | customer table

In [69]:
# 1. change col. name of customerid
df_customer.rename(columns={'customerid':'customer_id'},inplace=True)

In [70]:
# 2. change col name to customer_name
df_customer.rename(columns={'name':'customer_name'},inplace=True)

In [71]:
df_customer['gender'].unique()

array(['Male', 'Female', 'Women', 'Men'], dtype=object)

In [72]:
# 3. remove two incorrect values from gender table
df_customer['gender'].replace({'Men':'Male','Women':'Female'},inplace=True)

C:\Users\Yogesh\AppData\Local\Temp\ipykernel_16064\1547934463.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_customer['gender'].replace({'Men':'Male','Women':'Female'},inplace=True)


In [73]:
# 4. fix missing values in country table
df_customer[df_customer['country'].isnull()]

,customer_id,customer_name,country,state,gender,dob,interests,pincode
5,0013-MHZWF,durga,NaN,Delhi,Female,1988-12-10,NaN,NaN
8,0015-UOCOJ,maya,NaN,Kathmandu,Female,1985-07-07,NaN,NaN
12,0018-NYROU,chitra,NaN,Telangana,Female,2004-12-01,NaN,NaN


In [74]:
state_country_mapping = df_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()
df_customer['country'] = df_customer['country'].fillna(df_customer['state'].map(state_country_mapping))

In [75]:
df_customer[['country','state']]

,country,state
0,India,Maharashtra
1,India,Karnataka
2,India,Delhi
3,India,Nagaland
4,India,Delhi
...,...,...
6016,India,West Bengal
6017,India,Punjab
6018,India,Kerala
6019,India,Tamil Nadu


# data cleanning | subscription table

In [76]:
df_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaN,NaN,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaN,NaN,6.99,210,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaN,NaN,22.99,1725,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88


In [77]:
df_subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6021 entries, 0 to 6020
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               6021 non-null   object 
 1   subscription_start_date  6021 non-null   object 
 2   subscription_type        6021 non-null   object 
 3   renewal_date             5149 non-null   object 
 4   plan_type                6021 non-null   object 
 5   contract_type            6021 non-null   object 
 6   cancellation_date        878 non-null    object 
 7   cancellation_reason      878 non-null    object 
 8   monthly_charges          6021 non-null   float64
 9   cltv                     6021 non-null   int64  
 10  churn_score              6021 non-null   int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 517.6+ KB


In [ ]:
# 1 - change name of customerid to customer_id
# 2 - change dtype of all the date col to datetime

In [79]:
# A- change name of customerid to customer_id


df_subscription.rename(columns={'customerid' : 'customer_id'},inplace=True)

In [80]:
# B- change dtype of all the date col to datetime


date_col = ['subscription_start_date', 'renewal_date', 'cancellation_date']

df_subscription[date_col] = df_subscription[date_col].apply(pd.to_datetime)

# data cleanning | support table

In [81]:
df_support.head()

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28,N,60,NaN,service issue
1,0003-MKNFE,2024-08-28,Y,10,NaN,demaned refund
2,0013-EXCHZ,2024-01-20,Y,20,NaN,NaN
3,0013-MHZWF,2025-03-18,N,90,NaN,guidance to renew
4,0013-SMEOE,2024-11-01,N,30,NaN,NaN


In [82]:
df_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3863 entries, 0 to 3862
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerid      3863 non-null   object        
 1   complaint_date  3863 non-null   datetime64[ns]
 2   escalations     3863 non-null   object        
 3   csat_score      3863 non-null   int64         
 4   col_1           0 non-null      float64       
 5   comment         3858 non-null   object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 181.2+ KB


In [83]:
df_support.rename(columns={'customerid':'customer_id'},inplace=True)

In [84]:
# 1 - change name of customerid to customer_id
# 2 - drop col1 , comment

In [86]:
# A - change name of customerid to customer_id

df_support.rename(columns={'customerid':'customer_id'},inplace=True)

In [87]:
# B - drop col1 , comment


df_support.drop(columns=['col_1','comment'] ,inplace=True)

# feature engineering and data analysis

In [ ]:
#first fix support table duplicates then merge

In [93]:
#create a new col using existing col - churn flag

df_subscription['churn_flag'] = np.where(df_subscription['cancellation_date'].notna(),1,0)

In [94]:
df_subscription.shape

(6021, 12)

In [96]:
df_customer['customer_id'].nunique()

6021

In [98]:
df_subscription['customer_id'].nunique()

6021

In [99]:
df_support['customer_id'].size

3863

In [106]:
df_support['customer_id'].duplicated()

3736    False
1768    False
2684    False
1888    False
2961    False
        ...  
2308    False
2610    False
695     False
1776    False
2740    False
Name: customer_id, Length: 1952, dtype: bool

In [102]:
df_support

,customer_id,complaint_date,escalations,csat_score
0,0003-MKNFE,2024-08-28,N,60
1,0003-MKNFE,2024-08-28,Y,10
2,0013-EXCHZ,2024-01-20,Y,20
3,0013-MHZWF,2025-03-18,N,90
4,0013-SMEOE,2024-11-01,N,30
...,...,...,...,...
3858,9161-XUWBB,2022-11-04,N,96
3859,2633-PLFQC,2026-04-06,N,70
3860,2633-PLFQC,2026-07-05,N,64
3861,2633-PLFQC,2026-05-27,N,40


In [103]:
df_support['complaint_count'] = df_support.groupby('customer_id')['customer_id'].transform('count')
df_support = df_support.sort_values('complaint_date').drop_duplicates('customer_id', keep= 'last')

In [105]:
df_support.shape

(1952, 5)

In [107]:
df_support

,customer_id,complaint_date,escalations,csat_score,complaint_count
3736,3076-XJEVO,2020-10-08,N,57,2
1768,9762-BIFVO,2020-10-24,Y,34,1
2684,3440-MFQMO,2021-01-12,N,91,1
1888,7166-ICHTE,2021-01-19,N,74,1
2961,5723-PJOWZ,2021-01-29,N,79,1
...,...,...,...,...,...
2308,6557-INYVJ,2026-08-19,N,70,2
2610,5925-RHVQM,2026-08-19,N,74,3
695,6647-WGIVR,2026-08-19,N,81,1
1776,6990-BFLRU,2026-08-19,N,60,3


In [108]:
# merge
df = (df_subscription
                   .merge(df_customer, on ='customer_id', how= "left")
                   .merge(df_support, on = 'customer_id', how= "left"))

In [109]:
df.to_csv("cleaned churn data.csv", index=False)

# data analysis

In [110]:
df.head()

,customer_id,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,country,state,gender,dob,interests,pincode,complaint_date,escalations,csat_score,complaint_count
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,NaN,13.99,627,...,India,Maharashtra,Male,1982-04-12,travel,NaN,NaT,NaN,NaN,NaN
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,India,Karnataka,Male,1995-11-23,NaN,NaN,2024-08-28,Y,10.0,2.0
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,NaN,6.99,210,...,India,Delhi,Female,1978-02-15,movie,NaN,NaT,NaN,NaN,NaN
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,NaN,22.99,1725,...,India,Nagaland,Male,2001-08-30,NaN,NaN,NaT,NaN,NaN,NaN
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,...,India,Delhi,Female,1990-05-05,drama,NaN,2024-01-20,Y,20.0,1.0


In [111]:
df.columns

Index(['customer_id', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'interests', 'pincode', 'complaint_date', 'escalations', 'csat_score',
       'complaint_count'],
      dtype='object')

In [112]:
#1.  calculate churn rate

churn_rate = df['churn_flag'].mean() * 100
print('churn rate' , round(churn_rate,2), "%")

churn rate 14.58 %


In [113]:
#2. Retention rate

retention_rate = 100 - churn_rate
print('retention rate' , round(retention_rate,2), "%")

retention rate 85.42 %


In [114]:
#churn by plan type

churn_by_plan = df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index()

In [115]:
# 4 b. churn by state + sum(revenue) & and count of users

churn_by_state = df.groupby('state')['churn_flag'].mean().mul(100).round(2).reset_index()
revenue_by_state = df.groupby('state')['monthly_charges'].sum()
total_users_by_state = df.groupby('state')['customer_id'].count()

In [118]:
churn_by_state.head()

,state,churn_flag
0,Andhra Pradesh,15.26
1,Bihar,15.97
2,Delhi,14.29
3,Gujarat,13.01
4,Haryana,13.85


In [119]:
revenue_by_state.head()

state
Andhra Pradesh    5173.65
Bihar             5393.64
Delhi             5260.19
Gujarat           4878.31
Haryana           5152.31
Name: monthly_charges, dtype: float64

In [120]:
total_users_by_state.head()

state
Andhra Pradesh    367
Bihar             382
Delhi             371
Gujarat           346
Haryana           361
Name: customer_id, dtype: int64

In [121]:
# 4 c. churn by subscription type

churn_by_subscription_type = df.groupby('subscription_type')['churn_flag'].mean().mul(100).round(2).reset_index()

In [122]:
churn_by_subscription_type.head()

,subscription_type,churn_flag
0,Direct,14.30
1,Organic,0.00
2,Paid,14.82
3,Promotional,14.23
4,Refferal,14.24


In [123]:
# 5 ARPU - avetage revenue per user

ARPU = df['monthly_charges'].mean().round(2)

In [125]:
ARPU

np.float64(14.3)

In [126]:
df.columns

Index(['customer_id', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'interests', 'pincode', 'complaint_date', 'escalations', 'csat_score',
       'complaint_count'],
      dtype='object')

In [127]:
# 6 Average customer tenure
# count of days user has used our services : cancellation date else currunt date

today = pd.Timestamp.today()

df['tenure_days'] = np.where(
                df['cancellation_date'].notna(),
    
     (df['cancellation_date'] -  df['subscription_start_date']).dt.days,

     (today -  df['subscription_start_date']).dt.days,
)

avg_tenure = df['tenure_days'].mean()
print("Avg tenure (days) =", round(avg_tenure, 0))

Avg tenure (days) = 1016.0


In [128]:
# 7 revenue at risk - revenue lost from churned users

revenue_at_risk = df.loc[df['churn_flag']==1,'monthly_charges'].sum()
print("Revenue at risk (Rs 'K') =",revenue_at_risk)

Revenue at risk (Rs 'K') = 12424.23


In [129]:
# 8 Escalation rate

escalation_rate = (df['escalations']=='Y').mean()*100
("Escalation rate =", round(escalation_rate,2),"%")

('Escalation rate =', np.float64(4.97), '%')

In [130]:
# 9 average complaint per user

avg_complaints = df['complaint_count'].sum() / df['customer_id'].nunique()
print("Average complaint per user =", round(avg_complaints,2))

Average complaint per user = 0.64


In [131]:
# 10 corelation btw escalation vs churn

df['escalations'] = np.where(df['escalations'] == 'Y',1,0) # encoding str to int type
corr_df = df[['escalations','churn_flag']].dropna()

#correlation
correlation = corr_df['escalations'].corr(df['churn_flag'])
print("correlation between escalations and churn is = ", round(correlation,2))

correlation between escalations and churn is =  0.0


In [134]:
# churn risk - using existing col churn score
conditions = [
           (df['churn_score'] < 50 ),
           (df['churn_score'] >= 50) & (df['churn_score'] < 70),
           (df['churn_score'] >= 70)
]

risk_level = ['low','mid','high']

df['churn_risk'] = np.select(conditions, risk_level, default='unknown')

In [135]:
df[['churn_risk','churn_score']].tail()

,churn_risk,churn_score
6016,low,35
6017,low,15
6018,mid,51
6019,low,9
6020,low,26
